In [1]:
import logging

In [3]:
request_logger = logging.getLogger("../logs/K_log")
request_logger.setLevel(logging.INFO)

request_handler = logging.FileHandler("../logs/K_log.log", mode='w')
request_formatter = logging.Formatter("%(name)s %(asctime)s %(levelname)s %(message)s")

request_handler.setFormatter(request_formatter)
request_logger.addHandler(request_handler)

request_logger.info(f"Логгирование сборки датасета K_log...")

In [4]:
import pandas as pd
import time

In [5]:
GROUP_IDS = {
    "-211229778": ["Женский форум"                   , "zhenforum"       ],
    "-209976560": ["LABELCOM"                        , "labelcom"        ],
    "-203677279": ["Импроком"                        , "improcom"        ],
    "-165221845": ["ОХ"                              , "ox.show"         ],
    "-110135406": ["Азамат Мусугалиев"               , "azamatmusagaliev"],
    "-159848117": ["ТОП"                             , "top_show_top"    ],
    "-149430811": ["ВПИСКА"                          , "show_vpiska"     ],
    "-211437014": ["Артемий Лебедев"                 ,  "temalebedev"    ],
    "-64876876" : ["AcademeG //тру ориджинал групп//", "academeg_reviews"],
    "-131607049": ["ТНТ Видео"                       , "tnt_smotri"      ],
}

In [6]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('API_K.ipynb'), '..')))

from py.API_methods import get_all_videos, get_unique_album_videos, get_group_info, get_wall_info, get_video_comment

### Выгружаем основной датасет с видео

In [7]:
all_videos = []

for group, group_name in GROUP_IDS.items():
    request_logger.info(f"Достаем информацию из сообщества: {group_name[0]}...")

    request_logger.info(f"Получаем все видео из сообщества: {group_name[0]}...")
    videos = get_all_videos(group)

    total_videos = len(videos)
    request_logger.info(f"Всего видео: {total_videos}")

    time.sleep(1)
    
    request_logger.info(f"Получаем количество видео в альбомах из сообщества: {group_name[0]}...")
    album_videos = get_unique_album_videos(group)

    album_video_percent = round((album_videos / total_videos) * 100, 2) if total_videos > 0 else 0
    request_logger.info(f"Уникальных видео в альбомах: {album_videos} ({album_video_percent}%)")

    for video in videos:
        video["group_name"] = group_name[0]
        video["album_video_percent"] = album_video_percent
        all_videos.append(video)

    time.sleep(1) 

df = pd.DataFrame(all_videos)

cols_to_keep = [
    "id"        , "owner_id"   , "group_name" , "title"        , "description", "duration", "date",
    "views"     , "comments"   , "likes_count", "reposts_count", "player"     , "can_like",
    "can_repost", "can_dislike", "is_pinned"  , "image_url"    , "album_video_percent"    ,
]
existing_cols = [col for col in cols_to_keep if col in df.columns]
df = df[existing_cols]

In [8]:
df.shape

(4860, 15)

In [9]:
request_logger.info("Сохрнаяем итоговую таблицу в файл")
csv_filename = "../csv_files/videos.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

request_logger.info(f"Итоговый файл сохранён: {csv_filename}")

### Выгружаем доп инфу о сообещствах

In [10]:
group_ids = ",".join([value[1] for value in GROUP_IDS.values()])

optional_fields = ",".join([
    "description", "members_count", "activity", "status"    , "contacts", "ban_info",
    "links"      , "verified"     , "site"    , "age_limits", "banned"  , "city"    ,
    "country"    , "place"        ,
])

request_logger.info("Выгружаем доп информацию о сообещствах")
groups_data = get_group_info(group_ids, optional_fields)

if groups_data:
    request_logger.info("Успешно собрали")

    group_info = pd.DataFrame(groups_data.get("groups"))
    request_logger.info("Сохрнаяем итоговую таблицу в файл")

    csv_filename = "../csv_files/groups.csv"
    group_info.to_csv(csv_filename, index=False, encoding="utf-8")

    request_logger.info(f"Итоговый файл сохранён: {csv_filename}")
else:
    request_logger.warning("Информация о сообществах не собралась")

In [11]:
group_info.shape

(10, 20)

### Получаем посты сообществ

In [12]:
all_posts = []

for id, items in GROUP_IDS.items():
    request_logger.info(f"Получение постов для сообщества: {items[0]}")
    wall_posts_data = get_wall_info(id, items[1])

    if wall_posts_data:
        request_logger.info("Успешно собрали")
        all_posts.extend(wall_posts_data)
    else:
        request_logger.warning("Информация о постах не собралась")

    time.sleep(0.5)

if all_posts != []:
    request_logger.info("Успешно собрали все посты")

    posts = pd.DataFrame(all_posts)
    request_logger.info("Сохрнаяем итоговую таблицу в файл")

    csv_filename = "../csv_files/posts.csv"
    posts.to_csv(csv_filename, index=False, encoding="utf-8")

    request_logger.info(f"Итоговый файл сохранён: {csv_filename}")
else:
    request_logger.warning("Информация о постах не собралась")

In [13]:
posts.shape

(1010, 27)

### Получаем последний комментарий к видео

In [14]:
df_coments = pd.read_csv("../csv_files/videos.csv", usecols=['id', 'owner_id', 'title'])

In [15]:
df['top_comment_text'] = ""  

batch_size = 5
video_groups = [df.iloc[i:i + batch_size] for i in range(0, len(df), batch_size)]

for batch in video_groups:
    get_video_comment(batch, df)
    time.sleep(0.5)


# request_logger.info("Сохрнаяем итоговую таблицу в файл")
# csv_filename = "csv_files/comments.csv"
# df.to_csv(csv_filename, index=False, encoding="utf-8")

# request_logger.info(f"Итоговый файл сохранён: {csv_filename}")